# The Real $150,000 Was the Friends We Made Along the Way

The best public circuit for each task, checked and assembled into a submission, every time a better one is published. The headline below is this run's result; the working is hidden (click any cell to see it), and how it works is at the end.

Circuits come from our dataset [VQCC public circuits (Apache-2.0)](https://www.kaggle.com/datasets/manwithacat/vqcc-apache-public-circuits), credited to the notebooks that published them.

In [ ]:
def rank_cost(r):
    # Cost used to rank a manifest row within its task (lower is better); None if unscored. A DQI cost at one salt is
    # one draw of its sampled failures (one or two phase errors move R from 1.19126 to 1.19673 / 1.20139), so DQI
    # ranks by dqi_expected_cost (mean over a 48-draw sweep) when known, else dqi_zero_failure_cost (R floor x Q^2 x
    # mean Toffolis), else local_task_score (older manifests). invsqrt and chemistry rank by local_task_score.
    keys = ("dqi_expected_cost", "dqi_zero_failure_cost", "local_task_score") if r.get("task") == "dqi" \
        else ("local_task_score",)
    for k in keys:
        try:
            v = float(r.get(k) or "nan")
        except ValueError:
            continue
        if v == v:
            return v
    return None

def rank_basis(r):
    # Which column rank_cost used, for the tables
    if r.get("task") != "dqi":
        return "default salt"
    return ("expected (48 draws)" if r.get("dqi_expected_cost") else
            "zero-failure" if r.get("dqi_zero_failure_cost") else "default salt")

# Tie rule: rows within REL_TIE (relative) of the task's cheapest ranking cost are a tie, and the tie goes to the
# earliest public circuit (the original). A byte-different copy of a circuit changes the verifier's measurement
# draw, so it can come out a hair cheaper by noise alone; it should not displace the original's designer.
import datetime as _dt
REL_TIE = 1e-5

def public_when(r):
    # First non-empty, readable public_since / first_known / release_date as UTC (date only = 00:00 UTC); else None
    for k in ("public_since", "first_known", "release_date"):
        s = (r.get(k) or "").strip()
        if not s:
            continue
        try:
            d = _dt.datetime.fromisoformat(s.replace("Z", "+00:00"))
        except ValueError:
            continue
        return d.replace(tzinfo=_dt.timezone.utc) if d.tzinfo is None else d.astimezone(_dt.timezone.utc)
    return None

def rank_order(rows):
    # Scored rows of one task: the tie group (cost <= cheapest x (1 + REL_TIE)) earliest public first (missing dates
    # last, then lowest cost, then sha256), then the rest by cost, then sha256
    scored = [(rank_cost(r), r) for r in rows]
    scored = [(c, r) for c, r in scored if c is not None]
    if not scored:
        return []
    limit = min(c for c, _ in scored) * (1 + REL_TIE)
    far = _dt.datetime.max.replace(tzinfo=_dt.timezone.utc)
    def tie_key(cr):
        w = public_when(cr[1])
        return (w is None, w or far, cr[0], cr[1].get("sha256") or cr[1].get("sha12") or "")
    tied = sorted((cr for cr in scored if cr[0] <= limit), key=tie_key)
    rest = sorted((cr for cr in scored if cr[0] > limit),
                  key=lambda cr: (cr[0], cr[1].get("sha256") or cr[1].get("sha12") or ""))
    return [r for _, r in tied + rest]

def pick_best(rows_for_task):
    # The row to use for a task: rank_order()[0], or None if no row is scored
    order = rank_order(rows_for_task)
    return order[0] if order else None


import csv, glob, hashlib, json, shutil, sys, time, zipfile
from math import prod
from pathlib import Path

T0 = time.monotonic()
import numpy as np
import pandas as pd
from IPython.display import HTML, display

TASK_NAMES = {"invsqrt": "Inverse square root", "dqi": "DQI decoder", "chemistry": "Chemistry (dfthcsos)"}

def show(df, height=None):
    # A scrollable HTML table: readable on the notebook page, with live links
    html = df.to_html(index=False, render_links=True, escape=True, float_format=lambda x: f"{x:,.6g}")
    style = f"max-height:{height}px;overflow:auto;" if height else "overflow:auto;"
    display(HTML(f'<div style="{style}">{html}</div>'))

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
SUB = Path("/tmp/submission")   # assembled here; only submission.zip and check.json go to Output
WHEEL_GLOB = "kickmix-*-manylinux*x86_64*.whl"
# Organisers' reference costs (the score is the geometric mean of reference / ours over the three tasks)
REFERENCE = {"invsqrt": 2076774.233715282, "dqi": 24439463583672.93, "chemistry": 3355599588.3155375}
VERIFIER_TASK = {"invsqrt": "invsqrt", "dqi": "dqi", "chemistry": "dfthcsos"}

def find(pattern, root=INPUT):
    hits = sorted(Path(p) for p in glob.glob(f"{root}/**/{pattern}", recursive=True))  # follows symlinked mounts
    if not hits:
        raise FileNotFoundError(f"{pattern} not found under {root}")
    return hits

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(1 << 20):
            h.update(block)
    return h.hexdigest()

def family_sha256(directory):
    # The dataset's chemistry family hash: SHA-256 of sorted '<relative path>\t<file sha256>\n' lines
    files = sorted(p for p in Path(directory).rglob("*") if p.is_file())
    text = "".join(f"{p.relative_to(directory).as_posix()}\t{sha256(p)}\n" for p in files)
    return hashlib.sha256(text.encode()).hexdigest()

In [ ]:
manifest = find("manifest.csv")[0]
DATA = manifest.parent
rows = list(csv.DictReader(open(manifest, newline="")))
print(f"manifest: {manifest} ({len(rows)} circuits, sha256 {sha256(manifest)[:16]}...)")

picks = {}
for task in REFERENCE:
    scored = rank_order([r for r in rows if r["task"] == task])  # tie group first, original first
    picks[task] = scored[0]
    best_cost = min(rank_cost(r) for r in scored)
    ties = [r["sha12"] for r in scored[1:] if rank_cost(r) <= best_cost * (1 + REL_TIE)]
    print(f"{task:9} {len(scored):2} candidates; picked {scored[0]['sha12']} "
          f"ranking cost {rank_cost(scored[0]):.6e} ({rank_basis(scored[0])}); "
          f"default-salt cost {float(scored[0]['local_task_score']):.6e}"
          + (f"; ties within noise, earlier public kept: {', '.join(ties)}" if ties else ""))

# Chemistry: a verified assembly of public parts replaces the best single family only if it is cheaper
ASSEMBLY = None
best_json = DATA / "parts" / "assembly_best.json"
if best_json.is_file():
    a = json.loads(best_json.read_text())
    single = float(picks["chemistry"]["local_task_score"])
    print(f"parts/assembly_best.json: verified {a['verified_cost']:.6e} ({a['verify_date']}) "
          f"vs best single family {picks['chemistry']['sha12']} {single:.6e}")
    if a.get("verified_status") == "PASS" and float(a["verified_cost"]) < single:
        ASSEMBLY = a

shutil.rmtree(SUB, ignore_errors=True)
SUB.mkdir(parents=True)
for task, r in picks.items():
    if task == "chemistry" and ASSEMBLY:
        fam = SUB / "dfthcsos"
        fam.mkdir()
        files = [(ASSEMBLY["npz"]["dataset_path"], ASSEMBLY["npz"]["sha256"], "submission.npz")]
        files += [(s["dataset_path"], s["file_sha256"], f"angle_k_{s['slot']}.kmb") for s in ASSEMBLY["slots"]]
        for path, digest, name in files:
            assert sha256(DATA / path) == digest, f"{path} does not match parts/assembly_best.json"
            shutil.copyfile(DATA / path, fam / name)
        authors = sorted({s["author"] for s in ASSEMBLY["slots"]})
        picks["chemistry"] = {
            "task": "chemistry", "sha12": "assembled", "sha256": family_sha256(fam),
            "qubits": ASSEMBLY["qubits"], "local_task_score": str(ASSEMBLY["verified_cost"]),
            "author": "assembled: loaders " + ", ".join(authors) + "; weights " + ASSEMBLY["npz"]["author"],
            "notebook_title": "assembly of public chemistry parts (parts/assembly_best.json)",
            "source_notebook_url": "https://www.kaggle.com/datasets/manwithacat/vqcc-apache-public-circuits",
            "source_notebook_version": "", "release_date": ASSEMBLY["verify_date"],
            "core_author": ASSEMBLY["word_table_origin"]["author"],
            "change_summary": f"cheapest public loader per slot with the best weights on word table "
                              f"{ASSEMBLY['table_key']}", "also_found_in": ""}
        print(f"chemistry: assembled from {len(files)} dataset files, every SHA-256 checked")
    elif task == "chemistry":
        src = DATA / "circuits" / "chemistry" / r["sha12"]
        assert family_sha256(src) == r["sha256"], f"chemistry family {r['sha12']} does not match the manifest"
        shutil.copytree(src, SUB / "dfthcsos")
    else:
        src = DATA / "circuits" / task / f"{r['sha12']}.kmb"
        assert sha256(src) == r["sha256"], f"{task} {r['sha12']} does not match the manifest"
        shutil.copyfile(src, SUB / f"{task}.kmb")
print("assembled and hash-checked:", sorted(p.name for p in SUB.iterdir()))

pd.set_option("display.max_colwidth", 80)
display(pd.DataFrame([{"task": t, "candidates": sum(1 for x in rows if x["task"] == t and x["local_task_score"]),
                       "picked": r["sha12"], "qubits": r.get("qubits"),
                       "ranking cost": rank_cost(r), "ranked by": rank_basis(r),
                       "local cost (default salt)": float(r["local_task_score"]), "author": r["author"],
                       "sha256 checked": True} for t, r in picks.items()]).set_index("task"))

In [ ]:
tool_dirs = [Path(p).parent for root in ("/kaggle/usr/lib", "/kaggle/input")
             for p in sorted(glob.glob(f"{root}/**/kickmix_tools.py", recursive=True))]
assert tool_dirs, "attach the kickmix_tools utility script"
sys.path.insert(0, str(tool_dirs[0]))
import kickmix_tools as kt
print("kickmix_tools", kt.__version__, "from", tool_dirs[0])

wheel_dir = find(WHEEL_GLOB)[0].parent
FORK_DIR = kt.install(str(wheel_dir), target="/tmp/kickmix_fork")
FORK_PROVENANCE = kt.provenance()
print("fork provenance:", FORK_PROVENANCE)
print("competition data:", kt.competition_dir())

In [ ]:
PATHS = {"invsqrt": SUB / "invsqrt.kmb", "dqi": SUB / "dqi.kmb", "chemistry": SUB / "dfthcsos"}
results, check_rows = {}, []
for task, circuit in PATHS.items():
    r = kt.verify(VERIFIER_TASK[task], circuit, build="fork")
    results[task] = r
    res = r["result"]
    listed = float(picks[task]["local_task_score"])
    check_rows.append({
        "task": task, "circuit": picks[task]["sha12"], "status": res["status"],
        "cost": res["score"], "manifest cost": listed, "cost / manifest": res["score"] / listed,
        "times cheaper than reference": REFERENCE[task] / res["score"],
        "qubits": res.get("num_qubits"), "Toffolis (mean)": res.get("toffoli_count"),
        "value errors": res.get("value_errors") if task == "dqi" else None,
        "phase errors": res.get("phase_errors") if task == "dqi" else None,
        "wall s": round(r["wall_seconds"], 1),
        "peak MB": round(r["peak_pss_sum_mb"] or r["peak_rss_sum_mb"] or 0)})
checks = pd.DataFrame(check_rows).set_index("task")
FORMATS = {"cost": "{:.6e}", "manifest cost": "{:.6e}", "cost / manifest": "{:.9f}",
           "times cheaper than reference": "{:.4f}", "qubits": "{:.0f}", "Toffolis (mean)": "{:,.1f}",
           "value errors": "{:.0f}", "phase errors": "{:.0f}", "wall s": "{:.1f}", "peak MB": "{:,.0f}"}
shown = checks.copy()
for col, fmt in FORMATS.items():  # value/phase errors exist for DQI only; blank cells show as —
    shown[col] = [fmt.format(v) if pd.notna(v) else "—" for v in checks[col]]
display(shown)
print("— = not applicable: only the DQI verifier counts value and phase errors.")
failed = [t for t, r in results.items() if r["result"]["status"] != "PASS"]
assert not failed, f"not writing a submission: {failed} did not pass"

ratios = {t: REFERENCE[t] / results[t]["result"]["score"] for t in REFERENCE}
gm = prod(ratios.values()) ** (1 / 3)
# DQI cost depends on the draw. The headline uses the DQI ranking cost (mean over a 48-draw sweep, or the
# zero-failure cost); gm is this run's single default-salt draw (2026-10-31: the grader uses its own hidden draw).
ratios_exp = dict(ratios, dqi=REFERENCE["dqi"] / rank_cost(picks["dqi"]))
gm_exp = prod(ratios_exp.values()) ** (1 / 3)
zf = picks['dqi'].get('dqi_zero_failure_cost')  # best case: no failed DQI shots in a draw
gm_zero = prod(dict(ratios, dqi=REFERENCE['dqi'] / float(zf)).values()) ** (1 / 3) if zf else None
print(f"expected geometric mean: {gm_exp:.5f} (DQI {rank_basis(picks['dqi'])}); "
      f"this run's single default-salt draw: {gm:.5f}")
checks.to_csv(WORK / "checks.csv")

In [ ]:
members = [("invsqrt.kmb", SUB / "invsqrt.kmb"), ("dqi.kmb", SUB / "dqi.kmb")]
members += [(f"dfthcsos/{p.relative_to(SUB / 'dfthcsos').as_posix()}", p)
            for p in sorted((SUB / "dfthcsos").rglob("*")) if p.is_file()]
with zipfile.ZipFile(WORK / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name, path in members:
        z.write(path, name)
(WORK / "check.json").write_text(json.dumps({
    "manifest_sha256": sha256(manifest), "submission_sha256": sha256(WORK / "submission.zip"),
    "geometric_mean_local": gm, "fork": FORK_PROVENANCE,
    "picks": {t: {k: picks[t][k] for k in ("sha256", "author", "notebook_title", "source_notebook_url",
                                            "release_date", "local_task_score")} for t in picks},
    "chemistry_assembly": ASSEMBLY,
    "results": {t: r["result"] for t, r in results.items()}}, indent=1, default=str))
shutil.rmtree(SUB)
print("wrote", WORK / "submission.zip", "and", WORK / "check.json")

In [ ]:
# The page's headline: this run's portfolio and the top public circuits per task, from manifest.csv.
from datetime import datetime, timedelta, timezone
import html as _html

NOW = datetime.now(timezone.utc)
def _when(r):
    s = r.get("public_since") or r.get("release_date") or r.get("first_seen") or ""
    try:
        return datetime.fromisoformat(s if len(s) > 10 else s + " 12:00").replace(tzinfo=timezone.utc)
    except ValueError:
        return None
def _a(text, url):
    t = _html.escape(str(text))
    return f'<a href="{_html.escape(url)}" target="_blank">{t}</a>' if url else t

all_pass = all(r["result"]["status"] == "PASS" for r in results.values())
badge = ('<span style="color:#1a7f37;font-weight:600">all three verified (official verifier, PASS)</span>'
         if all_pass else '<span style="color:#cf222e;font-weight:600">VERIFICATION FAILED</span>')
range_line = (f'<div style="font-size:12px;color:#52514e;margin:2px 0 6px">A leaderboard score for this '
              f'portfolio usually differs from this in the third or fourth decimal place: a DQI draw with no '
              f'failed shots gives {gm_zero:.5f}, and one with more failures than average a little less than '
              f'{gm_exp:.5f}. That is expected.</div>'
              if gm_zero else '')
out = [f'<div style="font-size:15px;line-height:1.5">'
       f'<div style="font-size:30px;font-weight:700">Best public portfolio: {gm_exp:.5f}</div>'
       + range_line +
       f'<div style="color:#52514e">expected geometric mean, with DQI at its {rank_basis(picks["dqi"])} cost; '
       f'the single default-salt draw of this run gives {gm:.5f}. {badge}; run {NOW:%d %b %Y %H:%M} UTC; '
       f'{len(rows)} public circuits considered. The leaderboard uses its own hidden DQI draw, so a single '
       f'local draw is one sample, not a prediction.</div>'
       '<div style="font-size:11px;color:#8c8a85;margin-top:2px">The "Score … · Vn" label Kaggle shows '
       'on this page is the leaderboard result of the last submitted version (Vn), which can be older than '
       'this one.</div>']
for task in REFERENCE:
    cands = rank_order([r for r in rows if r["task"] == task])  # tie group first, original first
    tie_lim = min(rank_cost(r) for r in cands) * (1 + REL_TIE) if cands else 0
    top = []
    if task == "chemistry" and ASSEMBLY:
        top.append({"label": "assembly of public parts", "author": picks["chemistry"]["author"],
                    "url": "", "cost": float(ASSEMBLY["verified_cost"]), "qubits": ASSEMBLY["qubits"],
                    "when": None, "picked": True})
    for r in cands:
        if len(top) >= 5:
            break
        top.append({"label": r["sha12"], "author": r["author"], "url": r["source_notebook_url"],
                    "cost": rank_cost(r), "qubits": r["qubits"], "when": _when(r),
                    "date_only": len(r.get("public_since") or r.get("release_date") or r.get("first_seen") or "") <= 10,
                    "picked": picks[task]["sha12"] == r["sha12"], "tie": rank_cost(r) <= tie_lim})
    lead = ("; the next ties it within noise (the tie goes to the earliest public circuit)"
            if len(top) > 1 and top[1].get("tie") and top[0].get("tie") else
            f"; lead over the next: {top[1]['cost'] / top[0]['cost'] - 1:.3%}" if len(top) > 1 else "")
    out.append(f'<h3 style="margin:18px 0 4px">{TASK_NAMES[task]}: '
               f'{REFERENCE[task] / top[0]["cost"]:.4f}× cheaper than the reference{lead}</h3>')
    out.append('<table style="border-collapse:collapse"><tr style="color:#52514e;text-align:left">'
               '<th>#</th><th>circuit</th><th>published by</th><th>× cheaper</th><th>cost</th>'
               '<th>qubits</th><th>released</th><th></th></tr>')
    for i, t in enumerate(top, 1):
        new = (t["when"] is not None and NOW - t["when"] < timedelta(hours=24))
        tags = ("<b>in this submission</b> " if t["picked"] else
                "tie (within noise; later copy) " if t.get("tie") else "") + (
            '<span style="background:#fff3b0;padding:0 4px">NEW</span>' if new else "")
        weight = "font-weight:600;" if t["picked"] else ""
        when = ("—" if t["when"] is None else
                f"{t['when']:%d %b}" if t.get("date_only") else f"{t['when']:%d %b %H:%M} UTC")
        out.append(f'<tr style="{weight}border-top:1px solid #e8e7e2"><td>{i}</td>'
                   f'<td><code>{_html.escape(t["label"])}</code></td><td>{_a(t["author"], t["url"])}</td>'
                   f'<td>{REFERENCE[task] / t["cost"]:.4f}</td><td>{t["cost"]:.6g}</td><td>{t["qubits"]}</td>'
                   f'<td>{when}</td><td>{tags}</td></tr>')
    out.append("</table>")
out.append('<div style="color:#52514e;margin-top:10px">NEW = released in the last 24 hours. DQI cost = the '
           'ranking cost: the 48-draw expected cost where we have a sweep, else the zero-failure cost (see '
           '"How cheapest is chosen"). Full credits, the receipt and the frontier charts follow.</div></div>')
display(HTML("".join(out)))

## Receipt

One table that says what was submitted and why it can be trusted. Each gate is a check this run made, not a claim copied from elsewhere: the file is the one the manifest lists (by SHA-256; for an assembled chemistry entry, every file is the one `parts/assembly_best.json` lists), the unchanged official verifier passed it, and its cost matches the dataset's record of the same verifier on the same bytes. Saved as `receipt.json`.

In [ ]:
WALL = time.monotonic() - T0
receipt = pd.DataFrame([{
    "task": TASK_NAMES[t], "circuit": picks[t]["sha12"], "sha256": picks[t]["sha256"][:16] + "…",
    "published by": picks[t]["author"], "version": picks[t]["source_notebook_version"],
    "verdict": results[t]["result"]["status"],
    "times cheaper than reference": REFERENCE[t] / results[t]["result"]["score"],
    "link": picks[t]["source_notebook_url"]} for t in REFERENCE])
show(receipt)
CHEM_PARTS = None
if ASSEMBLY:
    print("Chemistry was assembled from public parts: each slot's loader, the weights and the word table's origin")
    CHEM_PARTS = pd.DataFrame(
        [{"part": f"slot {s['slot']} loader", "file": s["dataset_path"], "sha256": s["file_sha256"][:16] + "…",
          "published by": s["author"], "version": s.get("notebook_version", ""),
          "Toffolis per shot": s.get("toffoli_per_shot_source"), "link": s["notebook"]} for s in ASSEMBLY["slots"]]
        + [{"part": "weights (submission.npz)", "file": ASSEMBLY["npz"]["dataset_path"],
            "sha256": ASSEMBLY["npz"]["sha256"][:16] + "…", "published by": ASSEMBLY["npz"]["author"],
            "version": ASSEMBLY["npz"].get("notebook_version", ""), "Toffolis per shot": None,
            "link": ASSEMBLY["npz"]["notebook"]},
           {"part": f"word table {ASSEMBLY['table_key']} (origin)", "file": "parts/word_tables.csv",
            "sha256": "", "published by": ASSEMBLY["word_table_origin"]["author"], "version": "",
            "Toffolis per shot": None, "link": ASSEMBLY["word_table_origin"]["notebook"]}])
    show(CHEM_PARTS)
rel = {t: abs(results[t]["result"]["score"] / float(picks[t]["local_task_score"]) - 1) for t in REFERENCE}
GATES = {
    "G1 every file matches its listed SHA-256": True,  # asserted when the files were assembled
    "G2 the official verifier passed all three": all(r["result"]["status"] == "PASS" for r in results.values()),
    "G3 every cost within 1e-9 of the manifest record": max(rel.values()) <= 1e-9,
    "G4 notebook wall time under 2 hours": WALL <= 7200,
}
show(pd.DataFrame([{"gate": g, "result": "pass" if ok else "FAIL"} for g, ok in GATES.items()]))
print(f"this run's default-salt geometric mean {gm:.5f}; expected {gm_exp:.5f}; wall time {WALL / 60:.1f} min. "
      f"G3 means each cost was reproduced exactly at the default salt (the local verifier seeds every shot from "
      f"the file's SHA-256).")
print("Not a leaderboard prediction: the grader draws DQI shots with its own hidden salt (fixed per circuit, "
      "different from the local default), so its DQI failure count can differ from this run's. The expected "
      "geometric mean above is the fair estimate.")

# Headroom: how close each pick is to the verifier's pass/fail limits (from this run's verifier output)
E1_MAX = E2_MAX = 0.020          # Ha, both chemistry accuracy gates
GAP_FLOOR = 3.914898             # strict E_gap floor in the dfthcsos verifier
head = []
inv = results["invsqrt"]["result"]
w = inv.get("worst_error_over_tolerance")
if w is not None:
    head.append({"task": TASK_NAMES["invsqrt"], "check": "worst error / tolerance (must stay below 1)",
                 "value": f"{w:.5f}", "margin": f"{(1 - w) * 100:.3f}% of tolerance",
                 "tight": (1 - w) < 0.005})
dq = results["dqi"]["result"]
head.append({"task": TASK_NAMES["dqi"], "check": "failed shots on this file's draw (value / phase)",
             "value": f"{dq.get('value_errors')} / {dq.get('phase_errors')}",
             "margin": f"repetition factor R = {dq.get('repetition_factor', float('nan')):.5f}; "
                       "each extra failure costs about 0.46% of DQI cost",
             "tight": (dq.get("value_errors") or 0) + (dq.get("phase_errors") or 0) > 0})
ch = results["chemistry"]["result"].get("metrics", {})
for key, lim in (("error_1", E1_MAX), ("error_2", E2_MAX)):
    if key in ch:
        m = (lim - ch[key]) * 1000
        head.append({"task": TASK_NAMES["chemistry"], "check": f"{key} (must stay ≤ 20 mHa)",
                     "value": f"{ch[key] * 1000:.5f} mHa", "margin": f"{m:.5f} mHa", "tight": m < 0.01})
if "E_gap" in ch:
    m = ch["E_gap"] - GAP_FLOOR
    head.append({"task": TASK_NAMES["chemistry"], "check": "E_gap (must stay above 3.914898)",
                 "value": f"{ch['E_gap']:.7f}", "margin": f"{m:.2e}", "tight": m < 1e-4})
hd = pd.DataFrame(head)
hd["status"] = ["tight" if t else "comfortable" for t in hd.pop("tight")]
print("Headroom: how close each pick is to failing the verifier. 'tight' means a tiny change in the file, the "
      "verifier or floating point could flip it.")
show(hd)

# DQI final-salt risk for the picked file, from the DQI Salt Check notebook's output when it covers this file
SALT = None
for f in sorted(glob.glob(f"{INPUT}/**/dqi_salt_check_summary_*.csv", recursive=True)):
    for r in csv.DictReader(open(f, newline="")):
        if r.get("circuit") == picks["dqi"]["sha12"]:
            SALT = dict(r, file=Path(f).name)
if SALT:
    print(f"DQI final-salt risk for {picks['dqi']['sha12']} ({SALT['runs']} salts in {SALT['file']}): "
          f"errors per salt {SALT['errors per salt']}; P(0 failures) {SALT['P(0 err)']}; expected GM factor "
          f"{SALT['E[GM factor]']} (95% {SALT['GM factor 95%']}).")
else:
    print(f"DQI final-salt risk for {picks['dqi']['sha12']}: not measured yet (this file is not in the attached "
          f"DQI Salt Check output).")
if not GATES["G3 every cost within 1e-9 of the manifest record"]:
    print("G3 failed: this run's verifier disagrees with the manifest, so treat the local score with care.")
(WORK / "receipt.json").write_text(json.dumps({
    "gates": GATES, "geometric_mean_local": gm, "headroom": head, "dqi_salt_check": SALT, "wall_seconds": WALL, "relative_cost_difference": rel,
    "submission_sha256": sha256(WORK / "submission.zip"), "manifest_sha256": sha256(manifest),
    "rows": receipt.to_dict(orient="records"),
    "chemistry_parts": CHEM_PARTS.to_dict(orient="records") if CHEM_PARTS is not None else None},
    indent=1, default=str))

## How the public frontier developed

Every public circuit in the dataset, placed by when it became public and how much cheaper it is than the organisers' reference (higher is better). The blue staircase is the public frontier: each numbered marker is a circuit that beat every earlier one, numbered in the order it became public and shaded from light (early) to dark (latest). Only the current record is named; the largest jumps show their gain. Grey points are circuits that did not set a record. Where a circuit was made earlier in private and disclosed later, a hollow marker shows when it is first known to have existed, joined by a dotted line to the moment it became public; the public line itself is unchanged. The table under the charts lists every numbered step with who published it and the gain over the step before; it is also saved as `frontier_steps.csv`.

The second chart shows *where* each gain came from. Cost is a product of qubits and Toffolis, so the grey curves are lines of equal cost; a step to the left saved qubits, a step down saved Toffolis, and the arrows run from each record to the next. Chemistry circuits all hold about 430 qubits, so its chart shows Toffolis against λ_eff, the size of the factorized Hamiltonian's coefficients, instead. The animation after it replays the DQI and chemistry paths in public order, one record per frame (it is also saved as `frontier_path.gif`).

In [ ]:
from datetime import datetime, timezone

def parse_when(s):
    # 'YYYY-MM-DD', 'YYYY-MM-DD HH:MM[:SS]' or ISO with offset -> naive UTC datetime; None if blank or unreadable
    s = (s or "").strip()
    if not s:
        return None
    try:
        d = datetime.fromisoformat(s if len(s) > 10 else s + " 12:00")
    except ValueError:
        return None
    return d.astimezone(timezone.utc).replace(tzinfo=None) if d.tzinfo else d

def released(r):
    # When the circuit became public: public_since, falling back to release_date then first_seen (legacy rows)
    return (parse_when(r.get("public_since")) or parse_when(r.get("release_date"))
            or parse_when(r.get("first_seen")))

def chart_cost(r):
    # The cost the charts rank by: as the portfolio ranking (DQI: expected over 48 draws, else zero-failure, else the
    # default-salt score, because one DQI draw is luck; other tasks: the default-salt score)
    keys = ("dqi_expected_cost", "dqi_zero_failure_cost", "local_task_score") if r.get("task") == "dqi"             else ("local_task_score",)
    for k in keys:
        try:
            return float(r[k])
        except (KeyError, TypeError, ValueError):
            continue
    return None

def frontier_of(rows, task):
    # (every scored row of the task in public order, the rows that set a public record)
    rs = sorted((r for r in rows if r["task"] == task and chart_cost(r) and released(r)),
                key=lambda r: (released(r), chart_cost(r)))
    # A record must beat the previous one by more than the tie threshold (1e-5 relative, as REL_TIE in the ranking
    # rule): a republished copy that differs only by draw noise (e.g. 2366b67cf072, yoyobar's 67482db6c538 with
    # cancelling Clifford gates removed) is not a new record, and the original keeps the credit.
    steps, best = [], None
    for r in rs:
        c = chart_cost(r)
        if best is None or c < best * (1 - 1e-5):
            steps.append(r)
            best = c
    return rs, steps

def late_marks(rs):
    # Late disclosures: [(row, first_known, public)] for rows marked disclosure=late with a dated first_known
    # earlier than their public time. Drawn as a hollow marker at first_known joined by a dotted line to the public
    # point, so a hidden lead shows without rewriting the public history.
    out = []
    for r in rs:
        fk, pub = parse_when(r.get("first_known")), released(r)
        if (r.get("disclosure") or "") == "late" and fk and pub and fk < pub:
            out.append((r, fk, pub))
    return out

def date_locator(lo, hi, mdates):
    # About five ticks from the plotted span; fixed Day/Hour locators (AutoDateLocator warns on some spans)
    span_h = max((hi - lo).total_seconds() / 3600, 1)
    return (mdates.DayLocator(interval=max(1, round(span_h / 24 / 5))) if span_h >= 48
            else mdates.HourLocator(interval=max(1, round(span_h / 5))))

# Labelling and direction (user feedback 2026-10-30): the end point matters, the starting point does not. Records
# are numbered 1..n in public order (the same numbers as the 'step' column of the table); only the current best is
# named; a few of the largest jumps carry their gain; the first record is never named.
def step_gains(steps, value):
    # [(step number, value, gain over the previous record or None)] in public order
    out, prev = [], None
    for i, r in enumerate(steps, 1):
        v = value(r)
        out.append((i, v, v / prev - 1 if prev else None))
        prev = v
    return out

def label_plan(steps, value, max_gain_labels=3, min_gain=0.02):
    # {'end': index named with author and value, 'gains': indices whose jump is labelled '+x%'}; indices are 0-based
    # into steps. Index 0 (the starting point) is never named and has no gain; the end carries its own label.
    if not steps:
        return {"end": None, "gains": []}
    end = len(steps) - 1
    jumps = sorted(((g, i - 1) for i, _, g in step_gains(steps, value)[1:end] if g is not None and g >= min_gain),
                   reverse=True)[:max_gain_labels]
    return {"end": end, "gains": sorted(i for _, i in jumps)}

def number_priority(steps, value):
    # Order in which record numbers claim space when they would collide: the end, the labelled jumps, then the
    # rest from latest to earliest, so a crowded early cluster gives way first.
    plan = label_plan(steps, value)
    if plan["end"] is None:
        return []
    first = [plan["end"]] + plan["gains"][::-1]
    return first + [i for i in range(len(steps) - 1, -1, -1) if i not in first]

def fmt_cost(c):
    # Costs span 1e5 (invsqrt) to 1e11 (DQI): plain digits below ten million, three significant figures above
    return f"{c:,.0f}" if c < 1e7 else f"{c:.3g}".replace("e+", "e")

def end_label(r, value):
    # The one named label per panel: who holds the record now, how much cheaper than reference, and its cost
    return f"now: {r['author']}\n×{value(r):.3f}, cost {fmt_cost(chart_cost(r))}"

def ramp_colors(n, ramp):
    # n colours stepped evenly along a light->dark ramp: early records light, the latest darkest
    if n <= 1:
        return [ramp[-1]] * n
    return [ramp[round(k * (len(ramp) - 1) / (n - 1))] for k in range(n)]

def zoom_from(points, span_x, span_y, frac=0.3, min_n=4):
    # Index of the first record from which the rest of the path (at least min_n drawn points) fits within frac of
    # the chart's span on both axes: those records crowd a corner and get a magnified inset. None if none do.
    idx = [i for i, p in enumerate(points) if p is not None]
    for n, k in enumerate(idx[:max(0, len(idx) - min_n + 1)]):
        tx, ty = zip(*[points[i] for i in idx[n:]])
        if max(tx) - min(tx) <= frac * (span_x or 1) and max(ty) - min(ty) <= frac * (span_y or 1):
            return k
    return None

def path_segments(path):
    # path: [(step index, (x, y))] in record order. Consecutive (from, to) pairs, dropping zero-length moves (two
    # records at the same point: no arrow can point anywhere)
    return [(a, b) for a, b in zip(path, path[1:]) if a[1] != b[1]]


# Built only from manifest.csv: a new dataset version redraws the charts with no notebook change.
import base64
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
BLUE, GREY, INK, MUTED = "#2a78d6", "#b8b7b0", "#0b0b0b", "#52514e"
RAMP = ["#86b6ef", "#6da7ec", "#5598e7", "#3987e5", "#2a78d6", "#256abf", "#1c5cab", "#184f95"]  # blue 250..600
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": GREY,
                     "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
                     "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlecolor": INK,
                     "font.size": 10, "figure.dpi": 110})

def frontier(task):
    return frontier_of(rows, task)

def ratio(task):
    return lambda r: REFERENCE[task] / chart_cost(r)

def markers_px(ax):
    # Every plotted point on the axes (grey circuits and records), in display pixels: things labels avoid
    offs = [ax.transData.transform(c.get_offsets()) for c in ax.collections if len(c.get_offsets())]
    return np.vstack(offs) if offs else np.empty((0, 2))

def place_label(ax, xy, text, cands, avoid=(), fontsize=8.5, color=INK, **kw):
    # Put text beside xy at the first candidate offset (points) whose box covers no marker and no placed
    # box; if every candidate is blocked, the least crowded one. Returns the box in pixels.
    s = ax.figure.dpi / 72
    p = ax.transData.transform(xy)
    w, h = 0.6 * fontsize * max(len(t) for t in text.split("\n")) * s, 1.3 * fontsize * (text.count("\n") + 1) * s
    obst = [q for q in markers_px(ax) if np.hypot(*(q - p)) > 1]
    rend = ax.figure.canvas.get_renderer()
    frame = tuple(ax.get_window_extent(rend).extents)  # labels stay inside the plot area
    avoid = list(avoid) + [tuple(t.get_window_extent(rend).extents) for t in ax.texts if t.get_text()]
    best = None
    for dx, dy, ha in cands:
        x0 = p[0] + dx * s - (w if ha == "right" else w / 2 if ha == "center" else 0)
        y0 = p[1] + dy * s - h / 2
        box = (x0 - 2, y0 - 2, x0 + w + 2, y0 + h + 2)
        hits = sum(box[0] <= q[0] <= box[2] and box[1] <= q[1] <= box[3] for q in obst) + sum(
            not (box[2] < b[0] or b[2] < box[0] or box[3] < b[1] or b[3] < box[1]) for b in avoid)
        hits += 10 * (box[0] < frame[0] or box[1] < frame[1] or box[2] > frame[2] or box[3] > frame[3])
        if best is None or hits < best[0]:
            best = (hits, dx, dy, ha, box)
        if hits == 0:
            break
    _, dx, dy, ha, box = best
    ax.annotate(text, xy, xytext=(dx, dy), textcoords="offset points", ha=ha, va="center",
                fontsize=fontsize, color=color, zorder=6, **kw)
    return box, best[0]

NUM_SPOTS = [(0, 9, "center"), (0, -10, "center"), (7, 0, "left"), (-7, 0, "right")]

def place_numbers(ax, pts, order, avoid=()):
    # Number records in priority order (end first, earliest last) at a free spot beside each marker; a
    # number with no free spot is left off (the table under the charts has every step)
    placed = list(avoid)
    for i in order:
        if pts[i] is None:
            continue
        box, hits = place_label(ax, pts[i], str(i + 1), NUM_SPOTS, placed, fontsize=7.5)
        if hits:
            ax.texts[-1].remove()
            continue
        placed.append(box)

NOW_SPOTS = [(10, -12, "left"), (10, 10, "left"), (-10, 10, "right"), (-10, -12, "right"),
             (0, -22, "center"), (0, 20, "center")]

# 1. The frontier over time: numbered records on a staircase, the current record named at the end
fig, axes = plt.subplots(1, 3, figsize=(15, 4.9))
step_rows, later = [], []
for ax, task in zip(axes, REFERENCE):
    rs, steps = frontier(task)
    x, y = released, ratio(task)
    best_y = y(steps[-1])
    shown = [r for r in rs if y(r) >= 0.5 * best_y]  # off-scale outliers are listed in the credits table
    ax.scatter([x(r) for r in shown if r not in steps], [y(r) for r in shown if r not in steps],
               s=20, color=GREY, zorder=2, linewidths=0)
    xs = [x(r) for r in steps] + [max(x(r) for r in rs)]
    ys = [y(r) for r in steps] + [best_y]
    ax.step(xs, ys, where="post", color=BLUE, lw=1.8, zorder=3)
    cols = ramp_colors(len(steps), RAMP)
    ax.scatter([x(r) for r in steps], [y(r) for r in steps], s=46, c=cols, zorder=4,
               edgecolors="white", linewidths=1.5)
    late = [m for m in late_marks(rs) if y(m[0]) >= 0.5 * best_y]
    for r, fk, pub in late:  # hidden lead: hollow at first_known, dotted to the public point
        col = BLUE if r in steps else MUTED
        ax.plot([fk, pub], [y(r), y(r)], ls=":", lw=1.4, color=col, zorder=3)
        ax.scatter([fk], [y(r)], s=40, facecolors="white", edgecolors=col, linewidths=1.5, zorder=5)
    if late:
        ax.text(1.0, 1.02, f"{len(late)} disclosed late (hollow = first known)", transform=ax.transAxes,
                ha="right", fontsize=8, color=MUTED)
    plan = label_plan(steps, y)
    end = steps[plan["end"]]
    ax.set_title(TASK_NAMES[task], loc="left", pad=18)
    ax.text(0, 1.02, f"{len(steps)} records from {len(rs)} circuits", transform=ax.transAxes, fontsize=9,
            color=MUTED)
    ax.set_ylabel("× cheaper than reference (higher is better)")
    ax.grid(axis="y", color="#e8e7e2", lw=0.8)
    # About five ticks chosen from the plotted span, first-known markers included (fixed Day/Hour locators)
    loc = date_locator(min([x(r) for r in shown] + [fk for _, fk, _ in late]), max(x(r) for r in rs), mdates)
    ax.xaxis.set_major_locator(loc)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))
    pts = [(mdates.date2num(x(r)), y(r)) for r in steps]
    gains = [(pts[i], f"+{step_gains(steps, y)[i][2]:.1%}") for i in plan["gains"]]
    later.append((ax, pts, number_priority(steps, y), (pts[plan["end"]], end_label(end, y)), gains))
    for i, v, g in step_gains(steps, y):
        r = steps[i - 1]
        step_rows.append({"task": TASK_NAMES[task], "step": i,
                          "public since": r.get("public_since") or r["release_date"],
                          "first known": r.get("first_known", ""), "disclosure": r.get("disclosure", ""),
                          "circuit": r["sha12"], "published by": r["author"],
                          "core design by": r.get("core_author", ""), "qubits": r["qubits"],
                          "times cheaper than reference": v,
                          "gain over previous step": f"{g:+.2%}" if g is not None else "",
                          "what this file changes": r.get("change_summary", ""),
                          "link": r["source_notebook_url"]})
fig.suptitle("The public frontier over time: each numbered marker is a new record, darker = later",
             x=0.01, ha="left", fontsize=13, color=INK)
fig.tight_layout()
END_SPOTS = [(-8, -26, "right"), (-8, -44, "right"), (-30, -26, "right"), (-8, -62, "right"),
             (-8, -80, "right")]
GAIN_SPOTS = [(6, -11, "left"), (-6, -11, "right"), (6, 11, "left")]
for ax, pts, order, (end_xy, end_text), gains in later:  # labels placed after layout: collisions in final pixels
    avoid = [place_label(ax, end_xy, end_text, END_SPOTS)[0]]  # the current record, named
    avoid += [place_label(ax, xy, t, GAIN_SPOTS, avoid, fontsize=8, color=MUTED)[0] for xy, t in gains]
    place_numbers(ax, pts, order, avoid)
plt.show()

# 2. Where each gain came from: qubits against Toffolis, lines of equal cost, arrows from record to record
AXES = {"invsqrt": ("qubits", "Toffoli-equivalents per shot", lambda r, c: (int(r["qubits"]), c / int(r["qubits"]))),
        "dqi": ("qubits", "R × Toffolis per shot (thousands)", lambda r, c: (int(r["qubits"]), c / int(r["qubits"]) ** 2 / 1e3)),
        "chemistry": ("Toffolis", "λ_eff", lambda r, c: (float(r["T_total"]), float(r["lambda_eff"])))}

def equal_cost_curves(ax, task, best, steps, gx):
    q_chem = float(steps[-1]["Q_total"] or steps[-1]["qubits"]) if task == "chemistry" else None
    curves = []
    for k in (1.0, 1.1, 1.25, 1.45):
        cost = k * best
        gy = (cost / gx ** 2 / 1e3 if task == "dqi" else cost / gx if task == "invsqrt"
              else cost / (gx * q_chem))
        ax.plot(gx, gy, color="#e1e0da", lw=1, zorder=0)
        curves.append((k, gx, gy))
    return curves

def where_setup(ax, task):
    # Fixed frame for one task: equal-cost curves, limits, labels, and a magnified inset when the later
    # records crowd into a corner (zoom_from). Returns what the path drawing needs.
    rs, steps = frontier(task)
    xl, yl, f = AXES[task]
    best = chart_cost(steps[-1])
    near = [r for r in rs if chart_cost(r) <= 1.6 * best and (task != "chemistry" or r["T_total"])]
    pts = {r["sha12"]: f(r, chart_cost(r)) for r in near}
    px, py = zip(*pts.values())
    span = max(px) - min(px) or 1
    gx = np.linspace(min(px) - 0.5 * span, max(px) + 0.5 * span, 400)
    curves = equal_cost_curves(ax, task, best, steps, gx)
    pad_x, pad_y = 0.08 * (max(px) - min(px) or 1), 0.12 * (max(py) - min(py) or 1)
    at = [pts.get(r["sha12"]) for r in steps]
    k = zoom_from(at, max(px) - min(px), max(py) - min(py))
    ax.set_xlim(min(px) - pad_x, max(px) + pad_x * 2.2)  # room on the right for the curve labels
    y0, y1 = min(py) - pad_y, max(py) + pad_y
    ax.set_ylim(y0, y1 if k is None else y0 + (y1 - y0) / 0.48)  # with an inset: data in the lower half
    ax.set_xlabel(xl); ax.set_ylabel(yl)
    ax.set_title(TASK_NAMES[task], loc="left")
    ax.ticklabel_format(useOffset=False, style="plain", axis="both")
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    for c, gx_, gy_ in curves:  # label each equal-cost curve where it leaves the visible window
        inside = [(a, b) for a, b in zip(gx_, gy_) if x0 <= a <= x1 and y0 <= b <= y1]
        if inside:
            ax.annotate(f"{c:.2f}× best" if c > 1 else "best cost", inside[-1], fontsize=7.5, color=MUTED,
                        xytext=(2, 2), textcoords="offset points")
    ax.text(0.01, 0.97, "cheaper ↙", transform=ax.transAxes, fontsize=8.5, color=MUTED, va="top")
    hidden = len([r for r in rs if r not in near])
    if hidden:
        ax.text(1.0, 1.02, f"{hidden} costlier circuits not shown", transform=ax.transAxes,
                ha="right", fontsize=8, color=MUTED)
    inset = None
    if k is not None:
        tx, ty = zip(*[p for p in at[k:] if p is not None])
        sx, sy = (max(tx) - min(tx)) or 1, (max(ty) - min(ty)) or 1
        inset = ax.inset_axes([0.40, 0.53, 0.57, 0.42])
        equal_cost_curves(inset, task, best, steps, np.linspace(min(tx) - sx, max(tx) + sx, 200))
        inset.set_xlim(min(tx) - 0.12 * sx, max(tx) + 0.12 * sx)
        inset.set_ylim(min(ty) - 0.15 * sy, max(ty) + 0.15 * sy)
        inset.tick_params(labelsize=7)
        inset.ticklabel_format(useOffset=False, style="plain", axis="both")
        for s_ in ("top", "right"):
            inset.spines[s_].set_visible(True)
        inset.set_title(f"records {k + 1}–{len(steps)} magnified", fontsize=8, color=MUTED, loc="left",
                        fontweight="normal")
        ind = ax.indicate_inset_zoom(inset, edgecolor=MUTED, alpha=0.6)
        for c in (ind[1] if isinstance(ind, tuple) else getattr(ind, "connectors", None)) or []:
            c.set(alpha=0.3, lw=0.6)  # faint leaders: the box and the inset title carry the link
    return rs, steps, pts, inset, k

def where_draw(ax, task, rs, steps, pts, until=None, first=0):
    # Circuits public by `until` (all if None): grey others, the record path from record `first` on as
    # ramp-coloured arrows and markers. Returns (marker positions by step index, None where not drawn; the
    # records public by `until`).
    upto = [r for r in rs if until is None or released(r) <= until]
    rec = [r for r in steps if r in upto]
    rec_ids = {r["sha12"] for r in rec}
    other = [pts[r["sha12"]] for r in upto if r["sha12"] in pts and r["sha12"] not in rec_ids]
    if other:
        ax.scatter(*zip(*other), s=20, color=GREY, linewidths=0, zorder=2)
    cols = ramp_colors(len(steps), RAMP)
    at = [pts.get(r["sha12"]) for r in steps]  # None: record off this chart's window
    path = [(i, at[i]) for i in range(first, len(rec)) if at[i] is not None]
    for (i, a), (j, b) in path_segments(path):
        ax.annotate("", b, xytext=a, zorder=3, arrowprops=dict(arrowstyle="-|>", color=cols[j], lw=1.5,
                    mutation_scale=11, shrinkA=4, shrinkB=5))
    if path:
        ax.scatter([p[0] for _, p in path], [p[1] for _, p in path], s=46, c=[cols[i] for i, _ in path],
                   edgecolors="white", linewidths=1.5, zorder=4)
    return [at[i] if first <= i < len(rec) else None for i in range(len(steps))], rec

def where_panel(ax, task, until=None):
    # One panel: frame, path, and (when crowded) the inset. Returns [(axes, marker positions)] whose numbers
    # belong on that axes: records before the inset's first on the main axes, the rest in the inset.
    rs, steps, pts, inset, k = where_setup(ax, task)
    at, rec = where_draw(ax, task, rs, steps, pts, until)
    if inset is None:
        return [(ax, at)], steps, rec
    in_at, _ = where_draw(inset, task, rs, steps, pts, until, first=k)
    return [(ax, [p if i < k else None for i, p in enumerate(at)]), (inset, in_at)], steps, rec

fig, axes = plt.subplots(1, 3, figsize=(15, 4.9))
later = []
for ax, task in zip(axes, REFERENCE):
    parts, steps, _ = where_panel(ax, task)
    later.append((parts, number_priority(steps, ratio(task)), steps[-1]["author"]))
fig.suptitle("Where the gains came from: arrows run from each record to the next (down and left is cheaper)",
             x=0.01, ha="left", fontsize=13, color=INK)
fig.tight_layout()
for parts, order, who in later:  # the current record is named first, numbers fit around it
    for a, at in parts:
        end_xy = at[order[0]] if order else None
        avoid = [place_label(a, end_xy, f"now: {who}", NOW_SPOTS)[0]] if end_xy else []
        place_numbers(a, at, order, avoid)
plt.show()

# 3. The same DQI and chemistry paths replayed in public order, one record per frame, as a GIF (an <img> in
# the output, so it plays in the notebook viewer without JavaScript). The static chart above is the reference.
try:
    from matplotlib.animation import FuncAnimation, PillowWriter
    ANIM = ("dqi", "chemistry")
    fig = plt.figure(figsize=(10.5, 4.5), dpi=72)
    events = sorted((released(r), t, i) for t in ANIM for i, r in enumerate(frontier(t)[1]))
    HOLD = 6
    def frame(k):
        k = min(k, len(events) - 1)
        when, task, i = events[k]
        fig.clf()
        axes = fig.subplots(1, 2)
        fig.subplots_adjust(left=0.08, right=0.98, top=0.78, bottom=0.12, wspace=0.28)
        for ax, t in zip(axes, ANIM):
            parts, steps, rec = where_panel(ax, t, until=when)
            for a, at in parts:
                if t == task and at[i] is not None:  # ring the record that just arrived
                    a.scatter(*at[i], s=170, facecolors="none", edgecolors=INK, linewidths=1.2, zorder=5)
                place_numbers(a, at, number_priority(rec, ratio(t)) if rec else [])
        steps = frontier(task)[1]
        g = step_gains(steps, ratio(task))[i][2]
        fig.text(0.01, 0.95, f"{when:%d %b %H:%M} UTC   {TASK_NAMES[task]} record {i + 1} of {len(steps)}: "
                 f"{steps[i]['author']}" + (f", {g:+.1%}" if g is not None else ""),
                 fontsize=11, color=INK, ha="left")
        fig.text(0.01, 0.89, "arrows run from each record to the next; down and left is cheaper",
                 fontsize=9, color=MUTED, ha="left")
    anim = FuncAnimation(fig, frame, frames=len(events) + HOLD, interval=450)
    gif = WORK / "frontier_path.gif"
    anim.save(gif, writer=PillowWriter(fps=2.5))
    plt.close(fig)
    b64 = base64.b64encode(gif.read_bytes()).decode()
    display(HTML(f'<img src="data:image/gif;base64,{b64}" alt="DQI and chemistry frontier paths replayed '
                 f'in public order" style="max-width:100%">'))
    print(f"frontier_path.gif: {len(events)} records, {gif.stat().st_size / 1e6:.2f} MB (Output tab)")
except Exception as e:  # the static charts above carry the same information
    plt.close("all")
    print(f"animation skipped ({type(e).__name__}: {e})")

steps_df = pd.DataFrame(step_rows)
show(steps_df, height=420)
steps_df.to_csv(WORK / "frontier_steps.csv", index=False)

## Chemistry as parts

A chemistry entry is fifteen small circuits that each load one block of angle words, plus a factorization that turns those words into the Hamiltonian. Families that load the same words differ only in how cheaply each block is loaded (Toffolis) or how good the weights are (λ_eff), so the cheapest loader for every slot can be combined with the best weights, as long as each loader starts from the words the previous one left. Each small chart below follows one slot of the word table in use: the blue line is how many Toffolis the cheapest public loader for that slot has saved since the slot's first loader, and grey points are loaders as they appeared.

In [ ]:
# From parts/events.csv and parts/word_tables.csv: a new dataset version redraws it with no notebook change.
events_csv = DATA / "parts" / "events.csv"
if not events_csv.is_file():
    print("this dataset version has no parts/ folder: chart skipped")
else:
    ev = pd.read_csv(events_csv, dtype={"table_key": str, "id": str})
    tables = pd.read_csv(DATA / "parts" / "word_tables.csv", dtype={"table_key": str})
    key = ASSEMBLY["table_key"] if ASSEMBLY else tables.sort_values("rank").iloc[0]["table_key"]
    tinfo = tables.set_index("table_key").loc[key]
    lo = ev[(ev["kind"] == "loader") & (ev["table_key"] == key)].copy()
    lo["t"] = [released({"release_date": r, "first_seen": r}) for r in lo["release"]]
    slots = sorted(lo["slot"].astype(int).unique())
    ncol = 5
    nrow = -(-len(slots) // ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(15, 2.3 * nrow), sharex=True, sharey=True, squeeze=False)
    t_end = lo["t"].max()
    t_start = lo["t"].min()
    for ax, k in zip(axes.flat, slots):
        d = lo[lo["slot"].astype(int) == k].sort_values("t")
        first = d["toffoli"].iloc[0]
        best = d["toffoli"].cummin() - first  # Toffolis saved against the slot's first loader
        ax.axhline(0, color=GREY, lw=0.8, zorder=1)
        ax.scatter(d["t"], d["toffoli"] - first, s=16, color=GREY, linewidths=0, zorder=2)
        ax.step(list(d["t"]) + [t_end], list(best) + [best.iloc[-1]], where="post", color=BLUE, lw=1.8, zorder=3)
        ax.set_title(f"slot {k}", loc="left", fontsize=9, fontweight="normal", color=INK)
        ax.set_title(f"{d['toffoli'].min():.0f} ({best.iloc[-1]:+.0f})", loc="right", fontsize=9,
                     fontweight="normal", color=BLUE)
        ax.grid(axis="y", color="#e8e7e2", lw=0.8)
        # Fixed ticks (first loader, midpoint, now): the history spans hours, and AutoDateLocator with
        # so few ticks warns that it cannot pick an interval
        ax.set_xticks([t_start, t_start + (t_end - t_start) / 2, t_end])
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b\n%H:%M"))
    for ax in list(axes.flat)[len(slots):]:
        ax.set_visible(False)
    for ax in axes[:, 0]:
        ax.set_ylabel("Toffolis vs first", fontsize=8.5)
    fig.suptitle(f"Cheapest loader per slot over time, word table {key} ({tinfo['n_families']} families; words "
                 f"first published by {tinfo['first_author']}). Panel title: Toffolis per shot now (change)",
                 x=0.01, ha="left", fontsize=12, color=INK)
    fig.tight_layout()
    plt.show()
    now = lo.groupby("slot")["toffoli"].min()
    print(f"cheapest loaders on {key}: {now.sum():.0f} Toffolis per shot over {len(now)} slots "
          f"(first loaders on this table: {lo.sort_values('t').groupby('slot')['toffoli'].first().sum():.0f})")

## What didn't work

The frontier above is only half the story. [VQCC Negative Results](https://www.kaggle.com/datasets/manwithacat/vqcc-chemistry-negative-results) is a ledger of ideas tried on all three tasks and how each one failed or fell short, with its key numbers and the condition under which it would be worth reopening. It holds our own closed experiments and dead ends reported by other entrants, credited and linked. Check it before spending a day on an idea; if you have tried something that did not work, post it in the discussion and we will add it with credit. The [VQCC Negative Results notebook](https://www.kaggle.com/code/manwithacat/vqcc-negative-results) has filters and charts.

In [ ]:
# Reads ledger.csv from the attached VQCC Negative Results dataset; a new dataset version updates this with no
# notebook change. If the dataset is not attached, the links above still work.
neg_paths = [p for p in glob.glob(f"{INPUT}/**/ledger.csv", recursive=True)
             if "negative" in p.lower() and "parts" not in p.lower()]
if not neg_paths:
    print("VQCC Negative Results is not attached to this run; see the links above.")
else:
    neg = pd.read_csv(sorted(neg_paths)[0], dtype=str, keep_default_na=False)  # "null" is an outcome, not missing
    print(f"{len(neg)} entries: {(neg['source'] == 'ours').sum()} from our own experiments, "
          f"{(neg['source'] != 'ours').sum()} reported by others.")
    by = neg.pivot_table(index="task", columns="outcome", values="id", aggfunc="count", fill_value=0)
    show(by.reset_index())
    latest = neg.sort_values(["added_utc", "id"], ascending=False).head(10)
    print("Most recently added:")
    show(latest[["id", "task", "mechanism", "outcome", "source", "reopen_if", "link"]]
         .rename(columns={"reopen_if": "worth reopening if"}))

## Credits

The circuits used here are other people's public work, used unchanged under Apache 2.0. Each is credited to the notebook that published that exact file; where we have compared files, the credit also names who designed the circuit's core and what the publishing notebook changed. The second table lists every public circuit in the dataset, so you can scroll through the whole frontier and its authors; it is also saved as `credits.csv` (and `credits.parquet`) in the Output tab.

In [ ]:
# Everything here comes from manifest.csv, so a new dataset version updates the credits with no notebook change.
COLS = {"task": "task", "sha12": "circuit", "qubits": "qubits", "local_task_score": "local cost",
        "author": "published by", "notebook_title": "notebook", "source_notebook_version": "version",
        "release_date": "date", "core_author": "core design by", "core_design": "core design",
        "change_summary": "what this file changes", "source_notebook_url": "link"}

def credit_table(records):
    df = pd.DataFrame([{COLS[k]: r.get(k, "") for k in COLS} for r in records])
    df["local cost"] = pd.to_numeric(df["local cost"], errors="coerce")
    df["also used in"] = [len([u for u in (r.get("also_found_in") or "").split(";") if u]) for r in records]
    return df

print("The circuits in this submission:")
picked = credit_table([picks[t] for t in ("invsqrt", "dqi", "chemistry")])
picked["task"] = picked["task"].map(TASK_NAMES)
show(picked)
if CHEM_PARTS is not None:
    print("The chemistry entry is an assembly; its parts and their authors:")
    show(CHEM_PARTS)

ranked = sorted((r for r in rows if rank_cost(r) is not None),
                key=lambda r: (list(REFERENCE).index(r["task"]), rank_cost(r)))
everything = credit_table(ranked)
print(f"Every public circuit in the dataset ({len(everything)}), cheapest first within each task:")
show(everything, height=420)
everything.to_csv(WORK / "credits.csv", index=False)
try:
    everything.to_parquet(WORK / "credits.parquet", index=False)
except Exception as ex:  # parquet needs pyarrow or fastparquet
    print("credits.parquet not written:", ex)
print("Tooling: kickmix and the official verifiers by Google Quantum AI (their own terms); the kickmix fork, "
      "kickmix_tools and the circuits dataset by manwithacat, AI-assisted (Claude).")

## How this notebook works
> **This notebook is a compilation, not a method.** It adds no circuits or ideas of its own: it assembles the best public circuit for each task and checks it. If you want to analyse the public frontier, the underlying dataset [VQCC public circuits (Apache-2.0)](https://www.kaggle.com/datasets/manwithacat/vqcc-apache-public-circuits) is the better source: every public circuit, its local score, credits (who designed the core and what each notebook changed), and for chemistry the factorization file and its metrics. It gets a new version whenever a better public circuit appears.

**This version assembles itself.** Each run takes the cheapest public circuit for each task from our dataset [VQCC public circuits (Apache-2.0)](https://www.kaggle.com/datasets/manwithacat/vqcc-apache-public-circuits), checks every file against its SHA-256, re-verifies all three, and writes `submission.zip`. The dataset only holds circuits from public notebooks released under Apache 2.0, and its manifest credits each one to its source notebook; the credits printed below come straight from it. When a better circuit is added to the dataset, the notebook's "data updates" schedule rebuilds the portfolio.

**How the score is produced, in two steps:**
1. **Our check (this notebook).** We run the unchanged official verifiers on every circuit using our memory-lean [kickmix fork](https://github.com/manwithacat/kickmix-macos), loaded through our [`kickmix_tools` utility script](https://www.kaggle.com/code/manwithacat/kickmix-tools) and installed offline from the [wheels dataset](https://www.kaggle.com/datasets/manwithacat/kickmix-fork-wheels), which mirrors the fork's latest release. The fork gives the same results as the organisers' build with far less memory. The numbers printed below are *our* result.
2. **The official score.** When the `submission.zip` written here is submitted, Kaggle's grader runs the **official** verifier and kickmix build on it, with its own samples. That is the only score that counts; ours is a preview.

**How "cheapest" is chosen.** Each circuit in the manifest carries `local_task_score`: its cost from the official verifier run locally at the default salt. The tasks are scored independently, so the cheapest circuit of each task can come from a different notebook. invsqrt and chemistry are ranked by `local_task_score`. DQI is not: its cost is R × Q² × T, and R depends on how many of the 102,400 sampled shots fail (0 failures give R = 1.19126, one 1.19673, two 1.20139). One salt is one draw of that count, so ranking by it would reward luck rather than the circuit. DQI circuits are therefore ranked by `dqi_expected_cost`, the mean cost over a 48-draw salt sweep, when we have one, and otherwise by `dqi_zero_failure_cost`, R at zero failures × Q² × the mean Toffolis of the default-salt run. Circuits whose ranking costs are within 0.001% (relative 1e-5, about 25× the measured draw noise) of the task's cheapest are a tie, and the tie goes to the earliest public circuit, the original, so a byte-different copy cannot win on draw noise. The verification below still runs at the default salt, so the DQI cost printed there can sit above the ranking cost when this salt draws a failure; the grader draws its own. Our [salt-check notebook](https://www.kaggle.com/code/manwithacat/dqi-salt-check-public-frontier) measures how much DQI costs move between salts.

**Chemistry can be assembled from parts.** A chemistry entry is a set of fifteen small circuits (loaders) plus a factorization (weights). When the dataset's `parts/assembly_best.json` lists a cheaper verified combination of public loaders and weights than any single published family, this notebook builds that combination from the dataset files and credits every part's author.


**The steps (hidden cells above).** Pick the cheapest circuit per task from `manifest.csv` (or the chemistry parts assembly); check every file against its SHA-256; install our kickmix fork offline through `kickmix_tools` (it picks the wheel for this Python and checks it against the release's `SHA256SUMS`; each verification runs in a fresh subprocess with exactly one build on its import path); run the official verifiers at full official settings (default shot counts, the seed derived from each circuit's SHA-256, empty salt, the same settings as the manifest's `local_task_score`); write `submission.zip` with exactly the verified files, plus `check.json` and `checks.csv`.